In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_34_try_2.pickle

In [ ]:
%%PandasProfile
### cell 35 ###

# 1. Optimized helper to grab and clean subset of columns matching the question

def grab_subset_of_data_47(df, question_of_interest):
    return (
        df
        .filter(like=question_of_interest, axis=1)
        .rename(columns=lambda c: c.split('-')[-1].lstrip())
        .dropna(how='all')
    )

# 2. Optimized combiner for multiple years

def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_47(
    question_of_interest,
    include_2017=False
):
    # base mapping of year → source DataFrame
    mapping = [
        ('2018', responses_df_2018_cell10),
        ('2019', responses_df_2019_cell10),
        ('2020', responses_df_2020),
        ('2021', responses_df_2021),
        ('2022', responses_df_2022_cell10),
    ]
    if include_2017:
        mapping.append(('2017', responses_df_2017))
    # sort by year (string) to ensure chronological order
    mapping = sorted(mapping, key=lambda x: x[0])
    # build one combined DataFrame with a single concat + assign
    df_combined = pd.concat(
        (
            grab_subset_of_data_47(df_src, question_of_interest)
            .assign(year=year)
            for year, df_src in mapping
        ),
        ignore_index=True
    )
    # counts per year
    df_counts = df_combined.groupby('year', sort=True).count().reset_index()
    return df_combined, df_counts

# 3. Optimized converter from counts to percentages

def convert_df_of_counts_to_percentages_47(df, df_counts):
    years = df_counts['year'].astype(str)
    total_by_year = df['year'].value_counts().reindex(years).astype(int)
    pct = (
        df_counts.set_index('year')
                 .reindex(years)
                 .div(total_by_year, axis=0)
                 * 100
        ).reset_index()
    return pct

# 4. Prepare the question string and rename 2018 columns in place
question_of_interest_old = 'What machine learning frameworks have you used in the past 5 years?'
question_of_interest_new = 'Which of the following machine learning frameworks do you use on a regular basis?'
responses_df_2018_cell10.columns = (
    responses_df_2018_cell10.columns
        .str.replace(question_of_interest_old, question_of_interest_new, regex=False)
)
question_of_interest_cell47 = question_of_interest_new

# 5. Combine multi-year subsets and compute raw counts
ml_df_combined_cell47, ml_df_combined_counts_cell47 = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_47(
        question_of_interest_cell47
    )
)

# 6. Define groupings of the detailed framework columns into broader categories
groupings = {
    'TensorFlow/Keras': (
        ['TensorFlow', 'TensorFlow ', 'Keras', 'Keras '],
        'TensorFlow/Keras'
    ),
    'PyTorch/Lightning/Fast.ai': (
        ['PyTorch', 'PyTorch ', 'PyTorch Lightning ', 'Fast.ai ', 'Fastai'],
        'PyTorch/PyTorch Lightning/Fast.ai'
    ),
    'Xgboost/LightGBM/CatBoost': (
        ['Xgboost', 'Xgboost ', 'lightgbm', 'LightGBM ', 'catboost', 'CatBoost '],
        'Xgboost/LightGBM/CatBoost'
    ),
    'Scikit-learn': (
        ['Scikit—learn ', 'learn ', 'Learn'],
        'Scikit-learn'
    ),
}

# 7. Build a slim DataFrame with only 'year' + the new grouped columns
ml_df_slim = pd.DataFrame({'year': ml_df_combined_cell47['year']})
for new_col, (cols, label) in groupings.items():
    mask = ml_df_combined_cell47[cols].notna().any(axis=1)
    # mapping True→label, False→NaN automatically
    ml_df_slim[new_col] = mask.map({True: label})

# 8. Recompute counts by year for the grouped columns
ml_df_counts_grouped = (
    ml_df_slim
      .groupby('year', sort=True)[list(groupings.keys())]
      .count()
      .reset_index()
)

# 9. Convert counts to percentages
ml_df_percentages = convert_df_of_counts_to_percentages_47(
    ml_df_slim,
    ml_df_counts_grouped
)

# 10. Select and order the final columns, then pivot to long form
final_cols = ['year'] + list(groupings.keys())
ml_df_percentages = ml_df_percentages.loc[:, final_cols]

df_cell47 = (
    ml_df_percentages
      .melt(id_vars=['year'], var_name='', value_name='value')
      .sort_values(['year', 'value'], ascending=True)
)

df_cell47.info()